In [1]:
import os
import pandas as pd
import argparse
import sys

In [2]:
def check_table(table):
    red_color = "\033[01;31m"
    clear_color = "\033[01m"
    if(table.iloc[table.shape[0]-1,1]!="default"):
        sys.stderr.write(f"{red_color}The last row of table must be default values!{clear_color}\n")
        return False

In [5]:
def gen_assign(table, col):
    table_isnull = table.isnull()
    first_line = True
    code = "assign " + table.columns.values[col] + " = " 
    spaces = len(code) * ' '
    
    for row in range(table.shape[0]): # number of rows(insts)
        if (table_isnull.iloc[row,0] == False):
            #continue       # line commont
            if (table.iloc[row,0].startswith('//')): # line commont
                continue
        if(first_line == False):
            code += spaces
        first_line = False
        if(row != table.shape[0] - 1):
            if(table_isnull.iloc[row,col]):   # set to default value, the last row is default
                code += f"inst_type == {table.iloc[row,1]} ? {table.iloc[table.shape[0]-1,col]}" +" : \n"
            else:
                code += f"inst_type == {table.iloc[row,1]} ? {table.iloc[row,col]}" +" : \n" 
        else:
            if(table_isnull.iloc[row,col]):   # set to default value
                code += f"{table.iloc[table.shape[0]-1,col]};\n\n\n"
            else:
                code += f"{table.iloc[row,col]};\n\n\n"
    return code

In [6]:
table = pd.read_excel("exu_table.xlsx")
check_table(table)
table_isnull = table.isnull()
print(table_isnull.iloc[0,4])
print(table.iloc[table.shape[0]-1,3])
print(gen_assign(table, 4))

False
0
assign gpr_w_data = inst_type == `Inst_addi ? add_src1_imm : 
                    inst_type == `Inst_andi ? src1 & imm : 
                    inst_type == `Inst_xori ? src1 ^ imm : 
                    inst_type == `Inst_auipc ? idu_pc+imm : 
                    inst_type == `Inst_lui ? imm : 
                    inst_type == `Inst_jal ? idu_pc+4 : 
                    inst_type == `Inst_jalr ? idu_pc+4 : 
                    inst_type == `Inst_beq ? 0 : 
                    inst_type == `Inst_bne ? 0 : 
                    inst_type == `Inst_bgeu ? 0 : 
                    inst_type == `Inst_bltu ? 0 : 
                    inst_type == `Inst_blt ? 0 : 
                    inst_type == `Inst_bge ? 0 : 
                    inst_type == `Inst_or ? src1 | src2 : 
                    inst_type == `Inst_xor ? src1 ^ src2 : 
                    inst_type == `Inst_slt ? $signed(src1)<$signed(src2) ? 1 : 0 : 
                    inst_type == `Inst_sltu ? src1 < src2 ? 1 : 0 : 
        

In [7]:
def gen_all_assign(table):
    all_code = ""
    for col in range(2, table.shape[1]): #(output signals)
        all_code += gen_assign(table,col)
    return all_code

In [1]:
def gen_invalid_inst(table):
    table_isnull = table.isnull()
    code = "`ifndef STA\n"
    code_assign = "assign " + "exu_invalid_inst" + " = "
    code += code_assign
    spaces = len(code_assign) * ' '
    code += "rst == `RstEnable ? 0 : \n"
    code += spaces + "inst_type == `Inst_ebreak ? 0 : \n"
    
    for row in range(table.shape[0]): # number of rows(insts)
        if (table_isnull.iloc[row,0] == False):
            #continue       # line commont
            if (table.iloc[row,0].startswith('//')): # line commont
                continue
        code += spaces
        if(row != table.shape[0] - 1):
            code += f"inst_type == {table.iloc[row,1]} ? 0 : \n"
        else:
            code += "1;\n`endif\n\n\n"
    return code

In [9]:
table = pd.read_excel("exu_table.xlsx")
table

,commont,inst_type,gpr_we,gpr_w_addr,gpr_w_data,pc_offset_en,pc_offset,mem_re,mem_r_addr,mem_r_mask,mem_we,mem_w_addr,mem_w_data,mem_w_mask,sranum,csr_raddr,csr_we,csr_waddr,csr_wdata
0,NaN,`Inst_addi,`Enable,rd,add_src1_imm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,`Inst_andi,`Enable,rd,src1 & imm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,`Inst_xori,`Enable,rd,src1 ^ imm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,`Inst_auipc,`Enable,rd,idu_pc+imm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,`Inst_lui,`Enable,rd,imm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,`Inst_jal,`Enable,rd,idu_pc+4,`Enable,imm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,`Inst_jalr,`Enable,rd,idu_pc+4,`Enable,"{add_src1_imm[`WordWidth-1:1],1'b0} - idu_pc",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,`Inst_beq,NaN,NaN,NaN,`Enable,src1 == src2 ? imm : 4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,`Inst_bne,NaN,NaN,NaN,`Enable,src1 != src2 ? imm : 4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,`Inst_bgeu,NaN,NaN,NaN,`Enable,src1 >= src2 ? imm : 4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
print(gen_invalid_inst(table))

`ifndef STA
assign exu_invalid_inst = rst == `RstEnable ? 0 : 
                          inst_type == `Inst_ebreak ? 0 : 
                          inst_type == `Inst_addi ? 0 : 
                          inst_type == `Inst_andi ? 0 : 
                          inst_type == `Inst_xori ? 0 : 
                          inst_type == `Inst_auipc ? 0 : 
                          inst_type == `Inst_lui ? 0 : 
                          inst_type == `Inst_jal ? 0 : 
                          inst_type == `Inst_jalr ? 0 : 
                          inst_type == `Inst_beq ? 0 : 
                          inst_type == `Inst_bne ? 0 : 
                          inst_type == `Inst_bgeu ? 0 : 
                          inst_type == `Inst_bltu ? 0 : 
                          inst_type == `Inst_blt ? 0 : 
                          inst_type == `Inst_bge ? 0 : 
                          inst_type == `Inst_or ? 0 : 
                          inst_type == `Inst_xor ? 0 : 
                          inst_

In [11]:

#print(gen_invalid_inst(table))

In [12]:
if __name__ == "__main__":

    head_v = "code_head.v"
    tail_v = "code_tail.v"
    exu_table = "exu_table.xlsx"
    out_v = "exu.v"


    
    with open(head_v, 'r') as file:
        code_head = file.read()
        file.close()
    
    with open(tail_v, 'r') as file:
        code_tail = file.read()
        file.close()
    
    table = pd.read_excel(exu_table)
    code_invalid_inst = gen_invalid_inst(table)
    code_assign = gen_all_assign(table)
    code = code_head+code_invalid_inst+code_assign+code_tail
    print(code)
    # with open(out_v, 'w') as file:
    #     file.write(code_head+code_assign+code_tail)
    #     file.close()

`include "defines.v"
`include "inst_def.v"

/****************************************************
*
* Automatically generated file; DO NOT EDIT.
*
*****************************************************/

module exu(
        input clk,
        input rst,

        //from IDU
        input [`RegDataBus] src1,
        input [`RegDataBus] src2,
        input [`RegDataBus] imm,
        input [7:0] inst_type,
        input [`RegAddrBus] rd,
        input [`InstAddrBus] idu_pc,
        input [`InstDataBus] idu_inst,

        //to gpr
        output reg [`RegDataBus] gpr_w_data,
        output reg [`RegAddrBus] gpr_w_addr,
        output reg gpr_we,

        //to PC
        output reg [`InstAddrBus] pc_offset,
        output reg pc_offset_en,

        //to MEM
        output reg mem_we,
        output reg mem_re,
        output reg [`InstAddrBus] mem_w_addr,
        output reg [`InstAddrBus] mem_r_addr,
        output reg [`WordBus] mem_w_data,
        output reg [7:0] mem_w_mask,
        output